# Vulnerability modelling and probability of hail occurrence
Although the overall aim is to model occurrence of damage for the assets in question, e.g. a fleet of cars in an uncovered car park, it is (as always) desirable to quantify the probability of occurrence of hail and the damage to assets given occurrence; it is helpful to know to what extent damage is rare but severe or frequent and less severe. 

When defining probability of occurrence of hail where hail size can exceed a certain diameter, it is reasonable to use standard indices, especially those based on radar-derived measures such as Maximum Expected Severe Hail Size (MESHS). Radar data provides consistent spatial and temporal coverage and studies such as {cite:t}`schmid2023open` calibrate damage functions for buildings and cars in terms of MESHS parameters. However, even if a building is exposed to hail in the sense that the building is in a region that experiences a MESHS greater than a given threshold, that does not imply that the asset will be damaged by or even hit by hail of that diameter.

It is more generally the case that hazard indices tend to be cell-aggregated proxy measures as opposed to the point-level hazard-intensity exceedance curves more common when modelling, for example, flood. Moreover the intra-cell variability can be high. In principle a probability of occurrence can be inferred from hazard indices, but in practice damage functions are calibrated in terms of the observed indices.

We take an approach similar to {cite:t}`schmid2023open`. Two probabilities can then be defined:
- probability of occurrence of MESHS greater than a certain threshold (20 mm or 50 mm)
- probability that an asset is 'affected' by hail greater than a certain threshold

We choose the latter which is related to the 'PAA' measure of that paper, in that we model probability that an asset is 'affected'. This is, more precisely, the probability that a report of damage is made, assuming that the asset is insured. This is not a 'pure' probability of hail therefore, most notably because it may be that an asset is struck by hail but no damage is reported because it is too minor: either not observed or less than the deductible. <cite data-cite="schmid2025improved">Schmid et al. (2025)</cite> investigate the use of crowd-sourced data which could be used to estimate better the true hail probability. However, for the purposes of modelling damage, the probability affected by hail is appropriate as this can be combined with claims data in a transparent way. We choose this probability over probability of occurrence of MESHS greater than a certain threshold as a better approximation to a 'pure' probability: MESHS contains a significant fraction of false positives.

In principle we can calibrate the probability that an asset is affected from number of potential (i.e. meteorological conditions favourable for produciton of large hail) hail days using observed number of assets affected for a known set of assets over a known period.

We can then calculate the expected damage to an asset, given that the asset is expected to be affected. Defining $A$ to be an indicator of whether a certain asset is affected in a certain year and $M$ to be the MESHS of the hail event, we have:

$$
\mathbb{E}[\text{Damage} | A = 1] = \sum_{i=1}^N d(\bar{m}_i) P(m^{\text{min}}_i \le M < m^{\text{max}}_i | A = 1)
$$

$m^{\text{min}}_i$ and $m^{\text{max}}_i$ define the bounds of MESHS bin index $i$ and $\bar{m}$ is the bin mean. 

$d(m)$ is the damage sustained for a MESHS value, $m$, given that the asset is affected. This can be calculated as the average damage to affected assets as a fraction of the average asset value:

$$
d(m) = \frac{v_d / n_d}{v_e / n_e} = \frac{\text{MDR}(m)}{\text{PAA}(m)}
$$

- $v_d$ = damage cost
- $v_e$ = value of exposed assets
- $n_d$ = number of assets damaged
- $n_e$ = number of assets exposed


In [ ]:
import numpy as np

vt_build_paa, vh_build_paa, s_build_paa = 8.6, 66.9, 0.118
vt_build_mdr, vh_build_mdr, s_build_mdr = 20.0, 73.9, 1.47e-3

vt_car_paa, vh_car_paa, s_car_paa = 8.1, 55.0, 0.046
vt_car_mdr, vh_car_mdr, s_car_mdr = 13.7, 53.3, 3.96e-3


def f(v, vt, vh, s):
    vn = np.maximum(v - vt, 0) / (vh - vt)
    return s * vn**3 / (1 + vn**3)


# we define the damage functions, conditional on the asset being affected. The 'cond' subscript captures that condition.
def damage_cond_cars(x):
    return f(x, vt_car_mdr, vh_car_mdr, s_car_mdr) / f(
        x, vt_car_paa, vh_car_paa, s_car_paa
    )


def damage_cond_buildings(x):
    return f(x, vt_build_mdr, vh_build_mdr, s_build_mdr) / f(
        x, vt_build_paa, vh_build_paa, s_build_paa
    )

In [ ]:
# --- Figure 6a (Buildings): Number of damage reports by MESHS bin ---
fig6a_bin_min = np.array(
    [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95]
)
fig6a_bin_max = np.array(
    [25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
)
fig6a_n_reports = np.array(
    [
        7400,
        11800,
        17500,
        23100,
        20900,
        21100,
        22900,
        22800,
        22700,
        7500,
        4800,
        1000,
        100,
        150,
        300,
        100,
    ]
)

# --- Figure 7a (Cars): Fraction of damage reports by MESHS bin ---
fig7a_bin_min = np.array([20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85])
fig7a_bin_max = np.array([25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90])
fig7a_fraction = np.array(
    [
        0.038,
        0.049,
        0.065,
        0.068,
        0.081,
        0.097,
        0.082,
        0.072,
        0.051,
        0.023,
        0.016,
        0.008,
        0.001,
        0.001,
    ]
)

In [ ]:
n_buildings = 989000
n_years = 20
# probability that any building will generate a claim
prob = np.sum(fig6a_n_reports) / (n_buildings * n_years)
print(f"Annual probability of damage report: {prob:.5f}")

# probability that any building will generate a claim from a very large hail event, as determined by MESHS data
prob_50 = np.sum(fig6a_n_reports[fig6a_bin_min >= 50]) / (n_buildings * n_years)
print(f"Annual probability of damage report for MESHS > 50 mm: {prob_50:.5f}")

# damage given occurrence: break this probability into bins and give fractional damage per bin
prob_bins = fig6a_n_reports[0:14] / (n_buildings * n_years)
print(
    f"Bins (MESHS in mm): {list([int(min), int(max)] for min, max in zip(fig7a_bin_min, fig7a_bin_max))}"
)
print(
    f"Prob claim in bin (inferred from building data): {np.array2string(prob_bins, precision=3)}"
)
damage_bins_cars = damage_cond_cars((fig7a_bin_min + fig7a_bin_max) / 2)
damage_bins_cars = np.maximum.accumulate(damage_bins_cars)
print(
    f"Damage as a fraction of value of cars with claim: {np.array2string(damage_bins_cars, precision=3)}"
)
print(
    "It is assumed that for a car fleet, probability of an asset being affected is similar to that of a building. Car data is affected by sheltered cars. Even so, the number might be underestimated as a result of the deductible-related minimum."
)
expected_damage_cars = np.sum(damage_bins_cars * prob_bins)

# for cars:
print(
    f"Expected annual damage for a car fleet (as fraction of value): {np.array2string(expected_damage_cars, precision=5)}"
)
expected_damage_cond_cars = np.sum(damage_bins_cars * prob_bins) / np.sum(prob_bins)
print(
    f"Expected annual damage for a car fleet if asset affected (as fraction of value): {np.array2string(expected_damage_cond_cars, precision=4)}"
)

# for buildings:
damage_bins_buildings = damage_cond_buildings((fig7a_bin_min + fig7a_bin_max) / 2)
print(
    f"Damage as a fraction of value of buildings with claim: {np.array2string(damage_bins_buildings, precision=3)}"
)
expected_damage_cond_buildings = np.sum(damage_bins_buildings * prob_bins) / np.sum(
    prob_bins
)
print(
    f"Expected annual damage for a building if asset affected (as fraction of value): {np.array2string(expected_damage_cond_buildings, precision=5)}"
)

Annual probability of damage report: 0.00931
Annual probability of damage report for MESHS > 50 mm: 0.00416
Bins (MESHS in mm): [[20, 25], [25, 30], [30, 35], [35, 40], [40, 45], [45, 50], [50, 55], [55, 60], [60, 65], [65, 70], [70, 75], [75, 80], [80, 85], [85, 90]]
Prob claim in bin (inferred from building data): [3.741e-04 5.966e-04 8.847e-04 1.168e-03 1.057e-03 1.067e-03 1.158e-03
 1.153e-03 1.148e-03 3.792e-04 2.427e-04 5.056e-05 5.056e-06 7.583e-06]
Damage as a fraction of value of cars with claim: [0.033 0.053 0.067 0.078 0.085 0.089 0.091 0.092 0.092 0.092 0.092 0.092
 0.092 0.092]
It is assumed that for a car fleet, probability of an asset being affected is similar to that of a building. Car data is affected by sheltered cars. Even so, the number might be underestimated as a result of the deductible-related minimum.
Expected annual damage for a car fleet (as fraction of value): 0.00076
Expected annual damage for a car fleet if asset affected (as fraction of value): 0.0816
Dam

In [ ]:
import plotly.express as px

fig = px.scatter(
    x=(fig7a_bin_min + fig7a_bin_max) / 2,
    y=damage_bins_cars,
    labels={"x": "MESHS (mm)", "y": "Damage as fraction of TIV"},
    title="Car damage given car affected by hail",
)
fig2 = px.scatter(
    x=(fig7a_bin_min + fig7a_bin_max) / 2,
    y=damage_bins_buildings,
    labels={"x": "MESHS (mm)", "y": "Damage as fraction of TIV"},
    title="House damage given house affected by hail",
)

fig.update_traces(mode="lines+markers")
fig2.update_traces(mode="lines+markers")
fig.show()
fig2.show()